# Public-data audit before modeling
**Objective:** inspect provenance, target construction, timestamps and observation coverage. A public CSV still requires a research contract.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


In [ ]:
import hashlib
for name in ['external_manifest.json','brno_manifest.json','loed_manifest.json']:
    manifest=json.loads((ROOT/'data'/name).read_text())
    print(name,list(manifest)[:8])
loed=json.loads((ROOT/'data/loed_manifest.json').read_text())
for path,expected in loed['hashes'].items():assert hashlib.sha256((ROOT/path).read_bytes()).hexdigest()==expected
print('LoED:',loed['raw_rows'],'rows;',loed['files'],'files;',len(loed['utc_dates']),'UTC dates; license:',loed['license'])


## 1. A published derived target can be audited

In [ ]:
med=pd.read_csv(ROOT/'data/raw/medellin/measurements.csv',usecols=['ptx','gtx','ltx','grx','lrx','rssi','experimental_pl'])
expected=med.ptx+med.gtx-med.ltx+med.grx-med.lrx-med.rssi
print('Maximum identity discrepancy dB:',np.abs(expected-med.experimental_pl).max())
assert np.allclose(expected,med.experimental_pl,atol=1e-4)
del med


## 2. A timestamp column does not imply sorted, unique or regularly sampled data

In [ ]:
for path in sorted((ROOT/'data/raw/brno').glob('*.csv')):
    frame=pd.read_csv(path)
    print(path.name,'rows:',len(frame),'columns:',list(frame))
    timestamp=pd.to_datetime(frame['time'],unit='s',utc=True)
    print('Sorted:',timestamp.is_monotonic_increasing,'Repeated timestamps:',int(timestamp.duplicated().sum()))
    display(frame.head(3))


## 3. Inspect sentinels and coverage before selecting features

In [ ]:
sample=pd.read_csv(sorted((ROOT/'data/raw/loed').glob('*.csv'))[0],dtype={'device_address':str,'gateway':str,'mtype':str})
display(sample[['crc_status','spreading_factor','rssi','snr','size']].describe())
print('MTypes:',sample.mtype.value_counts().to_dict())
print('Recorded UTC dates:',pd.to_datetime(sample.time,utc=True).dt.date.unique())


**Exercise:** does a size value −1 mean a packet has negative length? Can six filenames be treated as six complete days?

<details><summary>Feedback after your attempt</summary>

a sentinel needs source interpretation and exclusion/imputation rules. Timestamps and capture coverage define the observation interval. The LoED sample spans eight UTC dates across six files; it is not a continuous six-day experiment.

</details>

**Adaptation task:** write a data card with source version, license, units, identifiers, coverage, sentinels, exclusions and a target the observations can actually support.